# Vesuvius Surface Detection - LB 0.549+ with Line Tracing

**Base**: [LB 0.549 Tuning](https://www.kaggle.com/code/baidalinadilzhan/lb-54-9-tuning) by Baidalin Adilzhan  
**Enhancement**: [Line tracing for filling holes](https://www.kaggle.com/code/hengck23/demo-for-line-tracing-for-filling-holes) by hengck23  
**Model**: TransUNet-SEResNeXt50 (160px, combo loss) by [ipythonx](https://www.kaggle.com/code/ipythonx/inference-vesuvius-surface-3d-detection)

## Key Improvements Over LB 0.549
1. **Line tracing post-processing**: Fills topological holes using Hungarian matching + Dijkstra pathfinding
2. **Targets TopoScore directly**: 30% of the metric is pure topology (Betti numbers)
3. **Probability-guided path filling**: Uses model confidence to trace optimal connections

## Metric Breakdown
```
Score = 0.30 * TopoScore + 0.35 * SurfaceDice@tau=2 + 0.35 * VOI_score
```
65% of the score is topology-related. Line tracing directly improves this.

---

## 0. Setup

In [ ]:
from IPython.display import clear_output

var = "/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
  "$var"/keras_nightly-*.whl \
  "$var"/tifffile-*.whl \
  "$var"/imagecodecs-*.whl \
  "$var"/medicai-*.whl \
  --no-index \
  --find-links "$var"

clear_output()
print('Setup complete.')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from medicai.transforms import Compose, NormalizeIntensity
from medicai.models import TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import zipfile
import tifffile
import heapq

import scipy.ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import convolve, gaussian_filter, label, generate_binary_structure
from skimage.morphology import remove_small_objects
from matplotlib import pyplot as plt

print(f'Keras backend: {keras.config.backend()}, version: {keras.version()}')

## 1. Configuration

In [ ]:
import glob

class CFG:
    # Paths
    root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
    test_dir = f"{root_dir}/test_images"
    output_dir = "/kaggle/working/submission_masks"
    zip_path = "/kaggle/working/submission.zip"
    
    # Model weights - try multiple known locations
    model_path = None
    _candidates = [
        "/kaggle/input/vsd-model/keras/transunet/3/transunet.seresnext50.160px.comboloss.weights.h5",
        "/kaggle/input/vsd-transunet-weights/transunet.seresnext50.160px.comboloss.weights.h5",
    ]
    
    # Model
    num_classes = 3
    input_shape = (160, 160, 160)
    
    # Sliding Window
    sw_overlap = 0.1
    sw_batch_size = 1
    
    # Post-processing (LB 0.549 params)
    T_low = 0.15
    T_high = 0.50
    z_radius = 3
    xy_radius = 1
    dust_min_size = 150
    
    # Line tracing (NEW)
    use_line_tracing = True
    lt_prob_threshold = 0.3
    lt_cc_dust = 100
    lt_w_len = 1.0
    lt_w_prob = 1.0
    lt_w_dir = 1.0
    lt_sigma_orient = 1.5
    lt_max_gap_dist = 50
    final_dust_min_size = 150

os.makedirs(CFG.output_dir, exist_ok=True)

# Find model weights
for path in CFG._candidates:
    if os.path.exists(path):
        CFG.model_path = path
        print(f"Found weights: {path}")
        break

if CFG.model_path is None:
    # Fallback: search all of /kaggle/input/
    candidates = glob.glob("/kaggle/input/**/*comboloss*.h5", recursive=True)
    if candidates:
        CFG.model_path = candidates[0]
        print(f"Found weights (search): {CFG.model_path}")
    else:
        raise FileNotFoundError("Model weights not found. Check dataset_sources.")

print(f'\nConfiguration:')
print(f'  Model: {CFG.model_path}')
print(f'  Post-processing: T_low={CFG.T_low}, T_high={CFG.T_high}, z={CFG.z_radius}, xy={CFG.xy_radius}')
print(f'  Line tracing: {"ENABLED" if CFG.use_line_tracing else "DISABLED"}')

## 2. Model

In [ ]:
model = TransUNet(
    input_shape=(*CFG.input_shape, 1),
    encoder_name='seresnext50',
    classifier_activation='softmax',
    num_classes=CFG.num_classes,
)
model.load_weights(CFG.model_path)

print(f'Model params: {model.count_params() / 1e6:.1f}M')

swi = SlidingWindowInference(
    model,
    num_classes=CFG.num_classes,
    roi_size=CFG.input_shape,
    sw_batch_size=CFG.sw_batch_size,
    mode='gaussian',
    overlap=CFG.sw_overlap,
)
print(f'Sliding window: overlap={CFG.sw_overlap}, mode=gaussian')

## 3. Preprocessing

In [ ]:
def load_volume(path):
    vol = tifffile.imread(path)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]  # (1, D, H, W, 1)
    return vol


def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        NormalizeIntensity(
            keys=["image"],
            nonzero=True,
            channel_wise=False
        ),
    ])
    result = pipeline(data)
    return result["image"]

print('Preprocessing functions defined.')

## 4. TTA Inference (7 views)

Returns both argmax class map AND foreground probability map.  
The probability map is needed for the line tracing step.

In [ ]:
def predict_with_tta(inputs, swi):
    """7-fold TTA: original + 3 flips + 3 rotations.
    
    Returns:
        class_map: (D, H, W) uint8 argmax class indices
        fg_probs:  (D, H, W) float32 foreground probability (class 1)
    """
    logits = []

    # Original
    logits.append(swi(inputs))

    # Flips (spatial only)
    for axis in [1, 2, 3]:
        img_f = np.flip(inputs, axis=axis)
        p = swi(img_f)
        p = np.flip(p, axis=axis)
        logits.append(p)

    # Axial rotations (H, W)
    for k in [1, 2, 3]:
        img_r = np.rot90(inputs, k=k, axes=(2, 3))
        p = swi(img_r)
        p = np.rot90(p, k=-k, axes=(2, 3))
        logits.append(p)

    mean_logits = np.mean(logits, axis=0)  # (1, D, H, W, C)
    
    # Extract foreground probability (class 1) before argmax
    fg_probs = mean_logits[0, ..., 1].astype(np.float32)  # (D, H, W)
    
    # Argmax for class map
    class_map = mean_logits.argmax(-1).astype(np.uint8).squeeze()  # (D, H, W)
    
    return class_map, fg_probs

print('TTA inference defined (7 views).')

## 5. Base Post-Processing (from LB 0.549)

In [ ]:
def build_anisotropic_struct(z_radius, xy_radius):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


def topo_postprocess(probs, T_low=0.15, T_high=0.50, z_radius=3, xy_radius=1, dust_min_size=150):
    """LB 0.549 post-processing: hysteresis + closing + dust removal."""
    # Step 1: 3D Hysteresis
    strong = probs >= T_high
    weak = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # Step 2: 3D Anisotropic Closing
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # Step 3: Dust Removal
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)

print('Base post-processing defined (LB 0.549 params).')

## 6. Line Tracing Post-Processing (NEW)

Fills topological holes in predicted surfaces by:
1. Finding broken curve segment start/end points per z-slice
2. Pairing them via Hungarian algorithm (optimal matching)
3. Connecting pairs via Dijkstra shortest path guided by the probability map

This directly targets the **TopoScore** (30% of metric) and **VOI** (35%) components.

In [ ]:
# ============================================================
# LINE TRACING HELPERS
# Credit: hengck23 - https://www.kaggle.com/code/hengck23/demo-for-line-tracing-for-filling-holes
# ============================================================

def find_segment_start_endpoint(skel):
    """Find start and end points of broken curve segments in a 2D binary mask.
    
    Start points: pixels with no neighbors in upper-left quadrant
    End points: pixels with no neighbors in lower-right quadrant
    """
    skel = (skel > 0).astype(np.uint8)
    
    # End points: no neighbors in lower-right
    kernel_end = np.array([
        0, 0, 0,
        0, 1, 1,
        0, 1, 1,
    ]).reshape(3, 3)
    neighbor_count = convolve(skel, kernel_end, mode='constant', cval=0) - skel
    endpoint_mask = (skel == 1) & (neighbor_count == 0)
    ey, ex = np.where(endpoint_mask)

    # Start points: no neighbors in upper-left
    kernel_start = np.array([
        1, 1, 0,
        1, 1, 0,
        0, 0, 0,
    ]).reshape(3, 3)
    neighbor_count = convolve(skel, kernel_start, mode='constant', cval=0) - skel
    startpoint_mask = (skel == 1) & (neighbor_count == 0)
    sy, sx = np.where(startpoint_mask)

    return (sy, sx), (ey, ex)


def pair_segment_start_endpoint(startpoint, endpoint, max_dist=50):
    """Pair start/end points via Hungarian algorithm with distance constraints."""
    sy, sx = startpoint
    ey, ex = endpoint

    start = np.column_stack([sy, sx])  # (Ns, 2)
    end = np.column_stack([ey, ex])    # (Ne, 2)

    Ns, Ne = len(start), len(end)
    if Ns == 0 or Ne == 0:
        return []

    dist = cdist(start, end, metric='euclidean')
    cost = dist.copy()

    INF = 1e6
    for i in range(Ns):
        for j in range(Ne):
            sy_i, sx_i = start[i]
            ey_j, ex_j = end[j]

            same_pixel = (sy_i == ey_j) and (sx_i == ex_j)
            forward = (ey_j >= sy_i) and (ex_j >= sx_i)

            if same_pixel:
                cost[i, j] = INF
            elif not forward:
                cost[i, j] = INF
            elif dist[i, j] > max_dist:
                cost[i, j] = INF  # Don't connect very distant points

    row_ind, col_ind = linear_sum_assignment(cost)

    pairs = []
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] >= INF:
            continue
        sy_i, sx_i = start[r]
        ey_j, ex_j = end[c]
        pairs.append((sy_i, sx_i, ey_j, ex_j))

    return pairs


def compute_orientation_field(prob, sigma=1.5):
    """Compute orientation field from probability map.
    
    Gradient points across ridges; we rotate 90 degrees to get along-ridge direction.
    This guides path-finding to follow the natural surface flow.
    """
    prob = prob.astype(np.float64)
    if sigma > 0:
        ps = gaussian_filter(prob, sigma=sigma)
    else:
        ps = prob
    gy, gx = np.gradient(ps)

    # Rotate 90 degrees: along-ridge direction
    vy = -gx
    vx = gy

    norm = np.sqrt(vx ** 2 + vy ** 2) + 1e-12
    vx /= norm
    vy /= norm

    return vy, vx


# 8-neighbor moves: (dy, dx, step_length)
NEIGH = [
    (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
    (-1, -1, np.sqrt(2.0)), (-1, 1, np.sqrt(2.0)),
    (1, -1, np.sqrt(2.0)), (1, 1, np.sqrt(2.0)),
]


def shortest_energy_path(startpoint, endpoint, prob, dir_y, dir_x,
                         w_len=1.0, w_prob=1.0, w_dir=1.0):
    """Find minimum-energy path between two points using Dijkstra.
    
    Cost combines:
    - Geometric length
    - Inverse probability (prefer high-confidence regions)
    - Direction coherence (prefer paths aligned with surface flow)
    """
    H, W = prob.shape
    sy, sx = startpoint
    ey, ex = endpoint

    p = prob.astype(np.float64)
    p = (p - p.min()) / (p.max() - p.min() + 1e-12)

    dist = np.full((H, W), np.inf, dtype=np.float64)
    prev_y = np.full((H, W), -1, dtype=np.int32)
    prev_x = np.full((H, W), -1, dtype=np.int32)

    pq = []
    dist[sy, sx] = 0.0
    heapq.heappush(pq, (0.0, int(sy), int(sx)))

    reached_end = None
    while pq:
        cur_cost, y, x = heapq.heappop(pq)
        if cur_cost > dist[y, x]:
            continue

        if (y, x) == (int(ey), int(ex)):
            reached_end = (y, x)
            break

        for dy, dx, step_len in NEIGH:
            ny, nx = y + dy, x + dx
            if ny < 0 or ny >= H or nx < 0 or nx >= W:
                continue

            c_len = w_len * step_len
            c_prob = w_prob * (1.0 - p[ny, nx])

            step_vec_y = dy / step_len
            step_vec_x = dx / step_len
            vy = dir_y[ny, nx]
            vx = dir_x[ny, nx]
            cosang = abs(step_vec_y * vy + step_vec_x * vx)
            c_dir = w_dir * (1.0 - cosang)

            new_cost = cur_cost + c_len + c_prob + c_dir

            if new_cost < dist[ny, nx]:
                dist[ny, nx] = new_cost
                prev_y[ny, nx] = y
                prev_x[ny, nx] = x
                heapq.heappush(pq, (new_cost, int(ny), int(nx)))

    if reached_end is None:
        return []

    # Backtrack
    path = []
    y, x = reached_end
    while not (y == -1 and x == -1):
        path.append((y, x))
        py, px = prev_y[y, x], prev_x[y, x]
        y, x = py, px
    path.reverse()
    return path


print('Line tracing helpers defined.')

In [ ]:
def line_trace_fill_holes(mask, fg_probs, cfg=CFG):
    """Apply line tracing to fill topological holes in the mask.
    
    For each connected component, process each z-slice:
    1. Find broken segment endpoints
    2. Pair them optimally (Hungarian algorithm)
    3. Connect via energy-minimizing path (Dijkstra)
    
    Args:
        mask: (D, H, W) uint8 binary mask from base post-processing
        fg_probs: (D, H, W) float32 foreground probabilities from TTA
        cfg: Configuration object
    
    Returns:
        filled_mask: (D, H, W) uint8 mask with holes filled
    """
    D, H, W = mask.shape
    filled = mask.copy()
    
    # Find connected components using scipy (26-connected = full 3x3x3 structure)
    struct = generate_binary_structure(3, 3)  # 26-connectivity
    cc, num_labels = label(mask, structure=struct)
    
    # Dust removal: remove small components
    for lbl in range(1, num_labels + 1):
        if np.sum(cc == lbl) < cfg.lt_cc_dust:
            cc[cc == lbl] = 0
    
    labels = np.unique(cc)
    labels = labels[labels != 0]  # Skip background
    
    total_filled = 0
    
    for label_id in labels:
        component_mask = (cc == label_id).astype(np.uint8)
        
        for z in range(D):
            slice_mask = component_mask[z]
            if slice_mask.sum() == 0:
                continue
            
            slice_prob = fg_probs[z]
            
            # Find start/end points
            (sy, sx), (ey, ex) = find_segment_start_endpoint(slice_mask)
            
            if len(sy) == 0 or len(ey) == 0:
                continue
            
            # Pair endpoints
            pairs = pair_segment_start_endpoint(
                (sy, sx), (ey, ex),
                max_dist=cfg.lt_max_gap_dist
            )
            
            if not pairs:
                continue
            
            # Compute orientation field once per slice
            dir_y, dir_x = compute_orientation_field(
                slice_prob, sigma=cfg.lt_sigma_orient
            )
            
            # Connect each pair
            for s_y, s_x, e_y, e_x in pairs:
                path = shortest_energy_path(
                    (s_y, s_x), (e_y, e_x),
                    slice_prob, dir_y, dir_x,
                    w_len=cfg.lt_w_len,
                    w_prob=cfg.lt_w_prob,
                    w_dir=cfg.lt_w_dir
                )
                
                for py, px in path:
                    if filled[z, py, px] == 0:
                        filled[z, py, px] = 1
                        total_filled += 1
    
    print(f'  Line tracing: filled {total_filled:,} voxels across {len(labels)} components')
    
    # Final dust removal
    if cfg.final_dust_min_size > 0:
        filled = remove_small_objects(
            filled.astype(bool), min_size=cfg.final_dust_min_size
        ).astype(np.uint8)
    
    return filled

print('Line tracing hole filling defined.')

## 7. Full Inference Pipeline

In [ ]:
def inference_pipeline(volume, cfg=CFG):
    """Complete pipeline: TTA -> base PP -> line tracing -> final cleanup.
    
    Returns:
        final_mask: (D, H, W) uint8 binary mask
        fg_probs: (D, H, W) float32 probabilities (for visualization)
    """
    # Step 1: TTA inference
    print('  Running 7-fold TTA...')
    class_map, fg_probs = predict_with_tta(volume, swi)
    
    # Step 2: Base post-processing (LB 0.549 params)
    print('  Applying base post-processing...')
    base_mask = topo_postprocess(
        class_map,
        T_low=cfg.T_low,
        T_high=cfg.T_high,
        z_radius=cfg.z_radius,
        xy_radius=cfg.xy_radius,
        dust_min_size=cfg.dust_min_size,
    )
    base_fg = base_mask.sum()
    print(f'  Base mask: {base_fg:,} foreground voxels ({base_fg/base_mask.size*100:.2f}%)')
    
    # Step 3: Line tracing hole filling
    if cfg.use_line_tracing:
        print('  Applying line tracing...')
        final_mask = line_trace_fill_holes(base_mask, fg_probs, cfg)
        final_fg = final_mask.sum()
        added = final_fg - base_fg
        print(f'  Final mask: {final_fg:,} voxels (added {added:,} via line tracing)')
    else:
        final_mask = base_mask
    
    return final_mask, fg_probs

print('Full inference pipeline defined.')

## 8. Create Submission

In [ ]:
test_df = pd.read_csv(f"{CFG.root_dir}/test.csv")
print(f'Test samples: {len(test_df)}')
print(f'Line tracing: {"ENABLED" if CFG.use_line_tracing else "DISABLED"}')
print()

with zipfile.ZipFile(CFG.zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for idx, image_id in enumerate(test_df["id"]):
        print(f'\n{"="*60}')
        print(f'[{idx+1}/{len(test_df)}] Processing: {image_id}')
        print(f'{"="*60}')
        
        tif_path = f"{CFG.test_dir}/{image_id}.tif"
        volume = load_volume(tif_path)
        print(f'  Shape: {volume.shape[1:-1]}')
        volume = val_transformation(volume)
        
        output, fg_probs = inference_pipeline(volume)
        
        # Save
        out_path = f"{CFG.output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, output.astype(np.uint8))
        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)
        
        # Free memory immediately
        del volume, fg_probs
        del output
        import gc; gc.collect()

print(f'\nSubmission ZIP: {CFG.zip_path}')

## 10. Submission Verification

In [ ]:
print('Submission verification:')
print(f'  ZIP: {CFG.zip_path}')

if os.path.exists(CFG.zip_path):
    zip_size = os.path.getsize(CFG.zip_path)
    print(f'  Size: {zip_size / 1024:.1f} KB')
    
    with zipfile.ZipFile(CFG.zip_path, 'r') as zf:
        files = zf.namelist()
        print(f'  Files: {len(files)}')
        for f in files:
            info = zf.getinfo(f)
            print(f'    {f}: {info.compress_size/1024:.1f} KB')
    
    expected = set(test_df['id'].astype(str).tolist())
    actual = set([f.replace('.tif', '') for f in files])
    
    if expected == actual:
        print('  All test IDs present. Submission valid!')
    else:
        print(f'  WARNING - Missing: {expected - actual}')